<a href="https://colab.research.google.com/github/himanshusar123/-ClaimVision-AI/blob/main/Day27_Multi_Agent_Helpdesk_Workflow_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import userdata

groq_api_key = userdata.get("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError(
        "GROQ_API_KEY was not found. Add it in Colab Secrets "
        "and enable notebook access."
    )

print("Groq API key loaded securely.")

Groq API key loaded securely.


In [3]:
%pip install -q -U "autogen-agentchat" "autogen-ext[openai]"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpcio-status 1.85.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorboard 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam-v1 0.14.6 re

In [11]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from google.colab import userdata

groq_api_key = userdata.get("GROQ_API_KEY")

model_client = OpenAIChatCompletionClient(
    model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key,
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": False,
        "family": "unknown",
        "structured_output": False
    }
)

print("Groq model client configured successfully!")

Groq model client configured successfully!


In [12]:
from autogen_core import CancellationToken

connection_test_agent = AssistantAgent(
    name="connection_test_agent",
    model_client=model_client,
    system_message="You are a concise technical assistant."
)

test_result = await connection_test_agent.run(
    task="Explain multi-agent orchestration in exactly three bullet points."
)

print(test_result.messages[-1].content)

- **Centralized control**: A manager schedules tasks, routes messages, and resolves conflicts among agents.  
- **Shared state management**: A broker or ledger synchronizes data, propagates events, and enforces consistency and policies.  
- **Automation framework**: Orchestration tools (e.g., Kubernetes, Airflow, custom middleware) deploy, scale, and recover agent workflows.


In [13]:
ticket_data = {
    "ticket_id": "TKT5001",
    "employee_name": "Amit Sharma",
    "employee_id": "EMP1001",
    "department": "Finance",
    "current_role": "Finance_Viewer",
    "requested_role": "Finance_Analyst",
    "manager_approval": "Pending",
    "issue": "Finance Reporting Access Denied",
    "priority": "High",
    "ticket_status": "Open",
    "access_request_status": "Pending Approval"
}

employee_question = (
    "Why can I not access the Finance Reporting Dashboard, "
    "and what happens next?"
)

print("Ticket loaded:", ticket_data["ticket_id"])
print("Employee question:", employee_question)

Ticket loaded: TKT5001
Employee question: Why can I not access the Finance Reporting Dashboard, and what happens next?


In [14]:
ticket_analyst = AssistantAgent(
    name="ticket_analyst",
    model_client=model_client,
    system_message="""
You are an IT Helpdesk Ticket Analyst.

Your tasks:
1. Read the supplied ticket data and employee question.
2. Identify the issue and summarize it clearly.
3. State the ticket ID, priority and current status.
4. Identify the requested role and current role.
5. Highlight that manager approval is Pending.
6. Do not invent missing information.
7. Do not claim that access has been granted.

Return a concise structured summary.
"""
)

analyst_result = await ticket_analyst.run(
    task=f"Analyze this verified ticket: {ticket_data}\n"
         f"Employee question: {employee_question}"
)

analyst_output = analyst_result.messages[-1].content
print(analyst_output)

**Ticket Summary**

- **Issue:** Finance Reporting Access Denied  
- **Ticket ID:** TKT5001  
- **Priority:** High  
- **Current Status:** Open  
- **Current Role:** Finance_Viewer  
- **Requested Role:** Finance_Analyst  
- **Manager Approval:** Pending  

**Next Steps:**  
The employee cannot access the Finance Reporting Dashboard because the role elevation to *Finance_Analyst* requires managerial approval, which is currently pending. Once the manager approves, the access request will move to the approval phase and the dashboard will become available.


In [15]:
knowledge_specialist = AssistantAgent(
    name="knowledge_specialist",
    model_client=model_client,
    system_message="""
You are an IT Helpdesk Knowledge Specialist.

Use only the ticket facts and the analyst summary supplied to you.
Provide safe, general troubleshooting suggestions such as:
- Confirm the employee is using the correct corporate account.
- Ask the employee to capture the exact error message and time.
- Suggest checking whether the dashboard is available to the employee's
  currently assigned role.
- Explain that role changes require the documented approval process.

Do not invent internal policies, URLs, contacts or technical findings.
Do not grant access or claim that approval is complete.
Clearly label suggestions as checks, not confirmed root causes.
"""
)

knowledge_result = await knowledge_specialist.run(
    task=f"Verified ticket data: {ticket_data}\n"
         f"Analyst summary: {analyst_output}\n"
         "Prepare safe troubleshooting guidance."
)

knowledge_output = knowledge_result.messages[-1].content
print(knowledge_output)

**Suggested Checks for Amit Sharma (Ticket TKT5001)**  

1. **Confirm Corporate Credentials**  
   - Verify that Amit is signing in with the correct corporate email/account linked to the Finance department.  

2. **Capture Exact Error Details**  
   - Ask Amit to take a screenshot or write down the precise error message (including any error code) and the exact time the access was attempted.  

3. **Check Role‑Specific Permissions**  
   - Verify whether the *Finance_Viewer* role is currently able to see the Finance Reporting Dashboard. If it is not, confirm that the dashboard is indeed restricted to the *Finance_Analyst* role.  

4. **Review Manager Approval Status**  
   - Since the manager approval is listed as “Pending,” remind Amit that the role elevation to *Finance_Analyst* cannot proceed until the manager’s formal approval is obtained.  

5. **Validate Current Access Rights**  
   - Confirm that the user has not been granted any temporary or conditional access that might interfe

In [16]:
response_writer = AssistantAgent(
    name="response_writer",
    model_client=model_client,
    system_message="""
You are a professional IT Helpdesk Response Writer.

Prepare an employee-facing response using only the verified ticket data,
the analyst summary and the knowledge specialist's suggestions.

Requirements:
- Address the employee by the supplied name.
- Include ticket ID, issue, priority and current status.
- Clearly explain that manager approval is Pending.
- Explain that access has not been confirmed as granted.
- Include relevant troubleshooting checks as suggestions.
- Do not promise a resolution time.
- Do not invent a manager decision, permission change or technical finding.
- Use a professional and reassuring tone.
"""
)

response_result = await response_writer.run(
    task=f"Verified ticket data: {ticket_data}\n"
         f"Analyst summary: {analyst_output}\n"
         f"Knowledge guidance: {knowledge_output}\n"
         "Draft the employee response."
)

draft_response = response_result.messages[-1].content
print(draft_response)

Subject: Update on Your Access Request – Ticket TKT5001

Hi Amit,

I’m writing to keep you informed about the status of your request for the Finance_Analyst role (ticket TKT5001).

**Ticket details**  
- **Employee:** Amit Sharma (EMP1001)  
- **Issue:** Finance Reporting Access Denied  
- **Priority:** High  
- **Current Status:** Open  
- **Current Role:** Finance_Viewer  
- **Requested Role:** Finance_Analyst  
- **Manager Approval:** Pending  
- **Access Request Status:** Pending Approval

**What this means for you**  
Your access to the Finance Reporting Dashboard cannot be granted until the manager’s formal approval is received. At this time, the approval is still pending, and access has not yet been confirmed as granted.

**Suggested next steps**  
While we await the approval, you can verify the following to help us identify any other possible blockers:

1. **Corporate credentials** – Make sure you’re logging in with your official corporate email/account that belongs to the Fina

In [17]:
async def run_helpdesk_workflow(ticket, question):
    print("\\n========== AGENT 1: TICKET ANALYST ==========")
    analyst = await ticket_analyst.run(
        task=f"Analyze ticket: {ticket}\nEmployee question: {question}"
    )
    analyst_text = analyst.messages[-1].content
    print(analyst_text)

    print("\\n========== AGENT 2: KNOWLEDGE SPECIALIST ==========")
    knowledge = await knowledge_specialist.run(
        task=f"Ticket: {ticket}\n"
             f"Analyst findings: {analyst_text}\n"
             "Provide safe troubleshooting guidance."
    )
    knowledge_text = knowledge.messages[-1].content
    print(knowledge_text)

    print("\\n========== AGENT 3: RESPONSE WRITER ==========")
    response = await response_writer.run(
        task=f"Verified ticket: {ticket}\n"
             f"Analyst findings: {analyst_text}\n"
             f"Knowledge guidance: {knowledge_text}\n"
             "Write the employee response."
    )
    response_text = response.messages[-1].content
    print(response_text)

    return {
        "analyst_summary": analyst_text,
        "knowledge_guidance": knowledge_text,
        "employee_response": response_text
    }

workflow_result = await run_helpdesk_workflow(
    ticket_data,
    employee_question
)

\n========== AGENT 1: TICKET ANALYST ==========
**Ticket Summary**

- **Ticket ID:** TKT5001  
- **Employee:** Amit Sharma (EMP1001) – Finance Department  
- **Issue:** Finance Reporting Access Denied  
- **Priority:** High  
- **Current Status:** Open  
- **Current Role:** Finance_Viewer  
- **Requested Role:** Finance_Analyst  
- **Manager Approval:** Pending  

**Explanation to Employee**

You are unable to access the Finance Reporting Dashboard because the role change to *Finance_Analyst* requires your manager’s approval. Once that approval is granted, the request will move from “Pending Approval” to the next stage and the dashboard will become accessible.
\n========== AGENT 2: KNOWLEDGE SPECIALIST ==========
**Safe Troubleshooting Guidance for Ticket TKT5001**

---

### 1. Confirm Login Credentials  
- **Action:** Verify that Amit is logging in with the correct corporate account that is linked to the Finance department.  
- **Why:** An incorrect or expired account could cause an “

In [18]:
def validate_helpdesk_result(ticket, result):
    checks = {
        "Ticket ID preserved":
            ticket["ticket_id"] in result["employee_response"],
        "Open status preserved":
            ticket["ticket_status"].lower() in
            result["employee_response"].lower(),
        "Pending approval preserved":
            "pending" in result["employee_response"].lower(),
        "No claim of granted access":
            not any(phrase in result["employee_response"].lower()
                    for phrase in [
                        "access has been granted",
                        "access is now granted",
                        "you now have finance analyst access"
                    ])
    }

    for check, passed in checks.items():
        print(("PASS" if passed else "REVIEW"), "-", check)

    return all(checks.values())

validation_passed = validate_helpdesk_result(
    ticket_data,
    workflow_result
)

print("Overall validation:", validation_passed)

PASS - Ticket ID preserved
PASS - Open status preserved
PASS - Pending approval preserved
PASS - No claim of granted access
Overall validation: True


In [19]:
def approval_checkpoint(ticket):
    if ticket["manager_approval"].lower() != "approved":
        return {
            "workflow_action": "PAUSED",
            "reason": "Manager approval is not complete.",
            "next_step": "Route the request to the authorized manager.",
            "access_change_performed": False
        }

    return {
        "workflow_action": "ELIGIBLE_FOR_AUTHORIZED_REVIEW",
        "reason": "Approval is recorded as approved.",
        "next_step": "Continue through the organization's authorized access process.",
        "access_change_performed": False
    }

approval_result = approval_checkpoint(ticket_data)
print(approval_result)

{'workflow_action': 'PAUSED', 'reason': 'Manager approval is not complete.', 'next_step': 'Route the request to the authorized manager.', 'access_change_performed': False}


In [20]:
final_report = {
    "ticket_id": ticket_data["ticket_id"],
    "employee": ticket_data["employee_name"],
    "priority": ticket_data["priority"],
    "ticket_status": ticket_data["ticket_status"],
    "manager_approval": ticket_data["manager_approval"],
    "workflow_validation_passed": validation_passed,
    "approval_checkpoint": approval_result["workflow_action"],
    "access_change_performed": False,
    "employee_response": workflow_result["employee_response"]
}

for key, value in final_report.items():
    print(f"{key}: {value}")

ticket_id: TKT5001
employee: Amit Sharma
priority: High
ticket_status: Open
manager_approval: Pending
workflow_validation_passed: True
approval_checkpoint: PAUSED
access_change_performed: False
employee_response: Subject: Update on Your Access Request – Ticket TKT5001

Hi Amit,

Thank you for reaching out about your access to the Finance Reporting Dashboard. Here’s the current status of your request:

| Detail | Information |
|--------|-------------|
| **Ticket ID** | TKT5001 |
| **Issue** | Finance Reporting Access Denied |
| **Priority** | High |
| **Current Status** | Open |
| **Current Role** | Finance_Viewer |
| **Requested Role** | Finance_Analyst |
| **Manager Approval** | Pending |
| **Access Request Status** | Pending Approval |

Because the requested role elevation to *Finance_Analyst* requires formal manager approval, the request remains in the “Pending Approval” state, and access has not yet been confirmed as granted.

While we wait for the manager’s approval, you may find 

In [21]:
await model_client.close()
print("Model client closed.")

Model client closed.


In [23]:
from google.colab import userdata
from autogen_ext.models.openai import OpenAIChatCompletionClient

groq_api_key = userdata.get("GROQ_API_KEY")

model_client = OpenAIChatCompletionClient(
    model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key,
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": False,
        "family": "unknown",
        "structured_output": False
    }
)

print("New Groq model client created successfully!")

New Groq model client created successfully!


In [24]:
from autogen_agentchat.agents import AssistantAgent

agent = AssistantAgent(
    name="Helpdesk_Assistant",
    model_client=model_client,
    system_message="""
    You are an IT Helpdesk Assistant.

    Explain technical issues clearly.
    Use only verified information.
    Never claim access has been granted without approval.
    """
)

print("Helpdesk Agent recreated successfully!")

Helpdesk Agent recreated successfully!


In [25]:
response = await agent.run(
    task="Explain the role of Redis and FastAPI in an IT Helpdesk system in three bullet points."
)

print(response.messages[-1].content)

- **FastAPI**: Acts as the web framework that exposes RESTful endpoints for ticket submission, status updates, and agent–user communication. Its asynchronous nature and automatic OpenAPI generation make it ideal for handling real‑time help‑desk interactions efficiently.  

- **Redis**: Serves as an in‑memory data store for caching frequently accessed information (e.g., ticket queues, user session data) and as a message broker for real‑time notifications (e.g., agent assignment alerts, chat messages). This reduces database load and speeds up response times.  

- **Integration**: FastAPI routes communicate with Redis to quickly read/write ticket state and push events, while a persistent database (e.g., PostgreSQL) stores long‑term records. The combination gives the help‑desk system low latency, scalability, and reliable real‑time updates.


In [26]:
security_reviewer = AssistantAgent(
    name="security_reviewer",
    model_client=model_client,
    system_message="""
You are a Helpdesk Security Review Agent.
Review the supplied draft for:
- Claims that access has been granted.
- Claims that manager approval is complete.
- Unsupported promises or invented facts.
Return PASS or NEEDS_REVISION and explain any issue.
Do not change permissions or approve requests.
"""
)

security_review = await security_reviewer.run(
    task=f"Verified ticket: {ticket_data}\nDraft: {draft_response}"
)

print(security_review.messages[-1].content)

PASS

**Explanation:**  
- The draft correctly states that access has **not** been granted and that manager approval remains **pending**.  
- No claims are made that the manager’s approval is complete or that the role elevation has already occurred.  
- All statements are supported by the ticket data; no unsupported promises or invented facts are present.


In [27]:
approved_ticket_simulation = ticket_data.copy()
approved_ticket_simulation["manager_approval"] = "Approved"

print("Original approval:", ticket_data["manager_approval"])
print("Simulation approval:", approved_ticket_simulation["manager_approval"])
print(approval_checkpoint(approved_ticket_simulation))

Original approval: Pending
Simulation approval: Approved
{'workflow_action': 'ELIGIBLE_FOR_AUTHORIZED_REVIEW', 'reason': 'Approval is recorded as approved.', 'next_step': "Continue through the organization's authorized access process.", 'access_change_performed': False}
